# Part 2 : Cortex AI ハンズオン

## このノートブックでやること
1. **名寄せ（エンティティ解決）** — AI_SIMILARITY で表記揺れを解消
2. **SNS ログの AI 分析** — AI_EXTRACT / AI_SENTIMENT / AI_CLASSIFY
3. **音声ログの AI 加工** — AI_REDACT / AI_SENTIMENT / AI_AGG

In [ ]:
%%sql -r dataframe_1
-- コンテキスト設定
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE GLACIERSTYLE_WH;
USE SCHEMA GLACIERSTYLE_DB.EC_ANALYTICS_SCHEMA;


---
## 1. 名寄せ（エンティティ解決）

仕入先から届いた商品リストと自社商品マスタを突合する処理です。  
実務では**「同じ商品なのに名前が微妙に違う」**ケースが頻発します。

**表記揺れの例**
- 全角/半角：`ＬＥＤデスクライト` → `LEDデスクライト`
- 同義語：`ライト` ↔ `ランプ` ↔ `灯`
- 語順逆転：`ウール 高品質ブランケット` → `高品質ウールブランケット`
- ブランド名付き：`GlacierStyle スタイリッシュデスクライト`

完全一致 JOIN では拾えない → **AI_SIMILARITY** で意味的な類似度を使います。

In [ ]:
%%sql -r dataframe_3
-- 仕入先データのプレビュー
SELECT
    supplier_product_id,
    supplier_product_name,
    supplier_name,
    supplier_price,
    supplier_category
FROM supplier_products_v2
LIMIT 10;


In [ ]:
%%sql -r dataframe_4
-- 商品マスタのプレビュー（突合先）
SELECT
    product_id,
    product_name,
    category_l2,
    brand,
    current_price
FROM dim_products
LIMIT 10;


![AI_SIMILARITY](images/part2/ai_similarity.png)

`AI_SIMILARITY` は **AI が意味的な類似度を計算**します。  
文字が全然違っても「同じ意味」なら高スコアになります。

> **スコアは 0〜1 の数値。高いほど意味が近く、1 が完全一致。**

**処理の考え方：**
- 仕入先商品（50件）× 商品マスタ（576件）= 全 28,800 通りの組み合わせを計算
- 各仕入先商品に対して「最もスコアが高いマスタ商品」を候補として選ぶ
- スコアが高い = AI が「似ている」と判断した組み合わせ  
  （最終的に「同じ商品か」の判断は人間が行う）

In [ ]:
%%sql -r dataframe_8
-- AI_SIMILARITY による名寄せ（サンプル50件）
-- 少し時間がかかります（CROSS JOIN のため）
--
-- 処理の流れ：
--   1. 仕入先商品（50件）× 商品マスタ（576件）を全組み合わせで比較
--   2. 各仕入先商品に対して最もスコアが高いマスタ商品を選ぶ（rank = 1）
--   3. スコアが高いほど AI が「同じ商品」と判断した組み合わせ
CREATE OR REPLACE TABLE work_ai_similarity_match AS
WITH sample_suppliers AS (
    SELECT * FROM supplier_products_v2 SAMPLE (50 ROWS)
),
similarity_calc AS (
    SELECT
        sp.supplier_product_id,
        sp.supplier_product_name,
        dp.product_id,
        dp.product_name,
        AI_SIMILARITY(sp.supplier_product_name, dp.product_name) AS ai_sim,
        ROW_NUMBER() OVER (
            PARTITION BY sp.supplier_product_id
            ORDER BY AI_SIMILARITY(sp.supplier_product_name, dp.product_name) DESC
        ) AS rank
    FROM sample_suppliers sp
    CROSS JOIN dim_products dp
)
SELECT
    supplier_product_id,
    supplier_product_name,
    product_id   AS matched_product_id,
    product_name AS matched_product_name,
    ROUND(ai_sim, 3) AS ai_similarity
FROM similarity_calc
WHERE rank = 1;

-- 結果確認（スコアが高い順）
SELECT * FROM work_ai_similarity_match
ORDER BY ai_similarity DESC;


---
## 2. SNS ログの AI 分析

SNS メンション（X / Instagram 等）の生データに AI 関数を適用します。

| 関数 | 処理内容 |
|---|---|
| `AI_EXTRACT` | テキストから商品名・カテゴリ・問い合わせタイプを抽出 |
| `AI_SENTIMENT` | 投稿の感情分析（ポジティブ / ネガティブ / ニュートラル） |
| `AI_CLASSIFY` | 投稿カテゴリ分類（称賛 / クレーム / 質問 / 提案） |

![AI_EXTRACT](images/part2/ai_extract.png)

![AI_SENTIMENT](images/part2/ai_sentiment.png)

![AI_CLASSIFY](images/part2/ai_classify.png)

**ポイント:** SELECT 文に関数を追加するだけで AI が動く。既存の SQL 資産をそのまま活用できる。

In [ ]:
%%sql -r dataframe_9b
-- 元データの確認（SNS 生ログ）
SELECT post_id, platform, username, content, posted_at
FROM raw_sns_mentions
LIMIT 10;


In [ ]:
-- SNS 生ログの AI 分析 → Gold 層テーブルへ保存
CREATE OR REPLACE TABLE gold_sns_mentions_analyzed AS
WITH extracted_data AS (
    SELECT
        post_id, platform, post_type, username, display_name,
        content, posted_at, likes, retweets, replies,
        hashtags, mentioned_products, media_urls,
        -- AI_EXTRACT: SNS投稿文から商品名・カテゴリ・問い合わせタイプを構造化データとして抽出する
        SNOWFLAKE.CORTEX.AI_EXTRACT(
            content,
            OBJECT_CONSTRUCT(
                'product_name',  'mentioned product name',
                'category',      'product category (e.g., ファッション, インテリア, テック)',
                'inquiry_type',  'inquiry type (e.g., 質問, レビュー, クレーム, 称賛, 提案)'
            )
        ) AS extracted_info
    FROM raw_sns_mentions
),
sentiment_analysis AS (
    SELECT *,
        -- AI_SENTIMENT: 投稿の感情をスコア化し、ポジティブ/ネガティブ/ニュートラルを判定する
        SNOWFLAKE.CORTEX.AI_SENTIMENT(content) AS sentiment_result
    FROM extracted_data
),
classified_data AS (
    SELECT *,
        -- AI_CLASSIFY: 投稿内容を「称賛/クレーム/質問/提案」の4カテゴリに自動分類する
        SNOWFLAKE.CORTEX.AI_CLASSIFY(
            content,
            ['称賛', 'クレーム', '質問', '提案']
        ) AS classification_result
    FROM sentiment_analysis
)
SELECT
    post_id, platform, post_type, username, display_name,
    content, posted_at, likes, retweets, replies,
    hashtags, mentioned_products, media_urls,
    -- 抽出結果
    extracted_info:response.product_name::VARCHAR  AS extracted_product_name,
    extracted_info:response.category::VARCHAR      AS extracted_category,
    extracted_info:response.inquiry_type::VARCHAR  AS inquiry_type,
    -- 感情分析結果
    sentiment_result:categories[0].name::VARCHAR   AS overall_sentiment,
    sentiment_result:categories[0].sentiment::VARCHAR AS sentiment,
    -- 分類結果
    classification_result:labels[0]::VARCHAR       AS post_category,
    CURRENT_TIMESTAMP()                            AS processed_at
FROM classified_data;


In [ ]:
-- 結果確認: 元テキストと各AI関数の分析結果を並べて表示
SELECT
    content,
    extracted_product_name,
    extracted_category,
    inquiry_type,
    sentiment,
    post_category
FROM gold_sns_mentions_analyzed
LIMIT 10;


---
## 3. 音声ログの AI 加工

コールセンター音声ログの文字起こしデータに AI 関数を適用します。

| 関数 | 処理内容 |
|---|---|
| `AI_REDACT` | 個人情報（氏名・電話番号・住所・クレカ番号）を自動マスキング |
| `AI_SENTIMENT` | 顧客感情の分析 |
| `AI_CLASSIFY` | 問い合わせカテゴリ分類 |
| `AI_AGG` | 通話内容を 400 文字以内に要約（GROUP BY 対応） |

![AI_REDACT](images/part2/ai_redact.png)

![AI_AGG](images/part2/ai_agg.png)

In [ ]:
%%sql -r dataframe_12
-- 音声ログの AI 加工 → Gold 層テーブルへ保存
CREATE OR REPLACE TABLE gold_voice_logs AS
SELECT
    * EXCLUDE transcribed_text,

    -- 1. AI_REDACT: 個人情報を自動マスキング
    SNOWFLAKE.CORTEX.AI_REDACT(transcribed_text)::VARCHAR AS transcribed_text_masked,

    -- 2. AI_SENTIMENT: 顧客感情の分析（マスキング済みテキストを使用）
    SNOWFLAKE.CORTEX.AI_SENTIMENT(transcribed_text_masked) AS sentiment_result,
    sentiment_result:categories[0].name::VARCHAR            AS overall_sentiment,
    sentiment_result:categories[0].sentiment::VARCHAR       AS sentiment,

    -- 3. AI_CLASSIFY: 問い合わせカテゴリ分類
    SNOWFLAKE.CORTEX.AI_CLASSIFY(
        transcribed_text_masked,
        ['商品に関する問い合わせ', '配送に関する問い合わせ', '返品・交換',
         '決済・支払い', 'アカウント・会員登録', 'クレーム', 'その他']
    ) AS classification_result,
    classification_result:labels[0]::VARCHAR               AS inquiry_category,

    -- 4. AI_AGG: 通話内容を 400 文字以内に要約（GROUP BY 対応）
    AI_AGG(
        transcribed_text_masked,
        '音声ログを400文字以内で要約してください。顧客の主な問い合わせ内容、要望、および解決状況を含めてください。'
    ) AS transcribed_text_summary,

    CURRENT_TIMESTAMP() AS processed_at

FROM raw_voice_logs
GROUP BY ALL;


In [ ]:
%%sql -r dataframe_13
-- 結果確認
SELECT * FROM gold_voice_logs LIMIT 10;


---
## まとめ

| やったこと | 使った関数 | ポイント |
|---|---|---|
| 名寄せ（完全一致） | JOIN | まずこれで問題を確認 |
| 名寄せ（AI） | `AI_SIMILARITY` | 同義語・語順逆転に強い |
| SNS 感情分析 | `AI_SENTIMENT` / `AI_CLASSIFY` | SELECT に追加するだけ |
| SNS 情報抽出 | `AI_EXTRACT` | 構造化されていないテキストから項目を取り出す |
| PII マスキング | `AI_REDACT` | 個人情報を自動検出して伏せ字に |
| 通話要約 | `AI_AGG` | GROUP BY 対応の集約 AI 関数 |

**次のパート (Part 3):** これらのデータを使って Snowflake Intelligence で自然言語分析を体験します。